<a href="https://colab.research.google.com/github/gspand/wc-wax-weather-app-20260210114014/blob/main/garmin/garmin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install garminconnect curl_cffi ua-generator
from garminconnect import Garmin

email = "gspandljan@gmail.com"
password = "JamesbondOO7"

api = Garmin(email, password)
api.login()

print("✅ Login erfolgreich")

In [ ]:
# ── SCHRITT 2: Tokens speichern und herunterladen ─────────────────────────────
# Diese Zelle direkt nach dem Login ausführen.
# garmin_tokens.zip wird automatisch heruntergeladen.
# In der App: Einstellungen → "Garmin-Token aus Colab hochladen" → ZIP hochladen.
# ─────────────────────────────────────────────────────────────────────────────
import shutil, os

token_dir = "/content/garmin_tokens_export"
os.makedirs(token_dir, exist_ok=True)
api.garth.dump(token_dir)

shutil.make_archive("/content/garmin_tokens", "zip", token_dir)

from google.colab import files
files.download("/content/garmin_tokens.zip")
print("✅ garmin_tokens.zip heruntergeladen")
print("→ App öffnen → Einstellungen → 'Garmin-Token aus Colab hochladen' → ZIP hochladen")
print("→ Dann 'Garmin-Daten importieren' → echte Touren erscheinen")

In [ ]:
print("Lade Aktivitäten...")

activities = []
start = 0

while True:
    batch = api.get_activities(start, 100)

    if not batch:
        break

    activities.extend(batch)
    start += len(batch)

    print(f"{len(activities)} geladen...")

print(f"Fertig. {len(activities)} Aktivitäten gefunden.")

Lade Aktivitäten...
100 geladen...
200 geladen...
300 geladen...
400 geladen...
500 geladen...
600 geladen...
700 geladen...
800 geladen...
900 geladen...
1000 geladen...
1100 geladen...
1200 geladen...
1300 geladen...
1302 geladen...
Fertig. 1302 Aktivitäten gefunden.


In [ ]:
from datetime import datetime
from collections import defaultdict
import time, json, math

TOUR_NAME = "Dolomiten Bikepacking 2026"
START_DATE = "2026-06-27"
OUTPUT_FILE = "bikepacking_svg.html"

COLORS = ["#e53935", "#1e88e5", "#43a047", "#8e44ad", "#fb8c00", "#00acc1"]

def parse_dt(s):
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def km(m): return round((m or 0) / 1000, 1)
def hm(m): return round(m or 0)

def fmt_h(sec):
    sec = sec or 0
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    return f"{h}:{m:02d} h"

def get_track_points(activity_id):
    details = api.get_activity_details(activity_id)
    pts = []

    for p in details.get("activityDetailMetrics", []):
        lat = p.get("directLatitude") or p.get("latitude")
        lon = p.get("directLongitude") or p.get("longitude")
        if lat is not None and lon is not None:
            pts.append([float(lat), float(lon)])

    if not pts:
        poly = details.get("geoPolylineDTO", {})
        for p in poly.get("polyline", []):
            lat = p.get("lat") or p.get("latitude")
            lon = p.get("lon") or p.get("longitude")
            if lat is not None and lon is not None:
                pts.append([float(lat), float(lon)])

    return pts

start_date = datetime.strptime(START_DATE, "%Y-%m-%d")

tour_activities = []
for a in activities:
    dt = parse_dt(a["startTimeLocal"])
    type_key = a.get("activityType", {}).get("typeKey", "")
    if dt >= start_date and type_key in ["road_biking", "cycling", "gravel_cycling", "mountain_biking"]:
        tour_activities.append(a)

tour_activities = sorted(tour_activities, key=lambda x: x["startTimeLocal"])

days = defaultdict(list)
for a in tour_activities:
    days[a["startTimeLocal"][:10]].append(a)

stages = []
all_pts = []

for idx, (day, acts) in enumerate(sorted(days.items())):
    color = COLORS[idx % len(COLORS)]
    pts_all = []

    for a in acts:
        print("Lade:", a["startTimeLocal"], a["activityName"])
        pts = get_track_points(a["activityId"])
        print("Trackpunkte:", len(pts))
        pts_all.extend(pts)
        time.sleep(0.2)

    all_pts.extend(pts_all)

    stages.append({
        "idx": idx + 1,
        "day": day,
        "color": color,
        "points": pts_all,
        "acts": acts,
        "title": acts[0].get("activityName", f"Tag {idx+1}"),
        "distance": km(sum(a.get("distance", 0) for a in acts)),
        "elevation": hm(sum(a.get("elevationGain", 0) for a in acts)),
        "moving": fmt_h(sum(a.get("movingDuration", 0) for a in acts)),
        "tss": round(sum(a.get("trainingStressScore", 0) or 0 for a in acts), 1)
    })

if not all_pts:
    raise Exception("Keine Trackpunkte gefunden – Route kann nicht gezeichnet werden.")

lats = [p[0] for p in all_pts]
lons = [p[1] for p in all_pts]

min_lat, max_lat = min(lats), max(lats)
min_lon, max_lon = min(lons), max(lons)

W, H = 1200, 800
PAD = 60

def project(lat, lon):
    x = PAD + (lon - min_lon) / (max_lon - min_lon) * (W - 2*PAD)
    y = H - PAD - (lat - min_lat) / (max_lat - min_lat) * (H - 2*PAD)
    return x, y

svg_paths = ""
stage_list = ""

for s in stages:
    if not s["points"]:
        continue

    coords = []
    step = max(1, len(s["points"]) // 1200)
    for lat, lon in s["points"][::step]:
        x, y = project(lat, lon)
        coords.append(f"{x:.1f},{y:.1f}")

    svg_paths += f"""
    <polyline points="{' '.join(coords)}"
              fill="none"
              stroke="{s['color']}"
              stroke-width="6"
              stroke-linecap="round"
              stroke-linejoin="round"/>
    """

    sx, sy = project(s["points"][0][0], s["points"][0][1])
    ex, ey = project(s["points"][-1][0], s["points"][-1][1])

    svg_paths += f"""
    <circle cx="{sx:.1f}" cy="{sy:.1f}" r="7" fill="{s['color']}"/>
    <circle cx="{ex:.1f}" cy="{ey:.1f}" r="7" fill="white" stroke="{s['color']}" stroke-width="4"/>
    """

    stage_list += f"""
    <div class="stage">
      <span class="dot" style="background:{s['color']}"></span>
      <div>
        <b>Tag {s['idx']} · {s['day']}</b><br>
        {s['title']}<br>
        <small>{s['distance']} km · {s['elevation']} hm · {s['moving']} · TSS {s['tss']}</small>
      </div>
    </div>
    """

total_km = round(sum(s["distance"] for s in stages), 1)
total_hm = sum(s["elevation"] for s in stages)
print(f"Fertig: {len(stages)} Etappen, {total_km} km, {total_hm} hm")

Lade: 2026-06-27 07:04:25 Graz Rennradfahren
Trackpunkte: 3227
Lade: 2026-06-28 07:45:11 Villach Rennradfahren
Trackpunkte: 2032
Lade: 2026-06-29 06:12:29 Arta Terme Rennradfahren
Trackpunkte: 2404
Lade: 2026-06-30 08:33:35 Auronzo di Cadore Rennradfahren
Trackpunkte: 2350
Lade: 2026-07-01 07:46:53 Livinallongo del Col di Lana Rennradfahren
Trackpunkte: 3020
Lade: 2026-07-03 14:44:53 Innsbruck Rennradfahren
Trackpunkte: 3310
Lade: 2026-07-04 09:11:54 Innsbruck Rennradfahren
Trackpunkte: 2396
Lade: 2026-07-05 08:04:09 Landeck Rennradfahren
Trackpunkte: 3246
Lade: 2026-07-06 08:54:07 Feldkirch Rennradfahren
Trackpunkte: 1960
Lade: 2026-07-08 07:50:36 Davos Rennradfahren
Trackpunkte: 3701
Lade: 2026-07-09 07:23:08 Chiavenna Rennradfahren
Trackpunkte: 2826
Lade: 2026-07-10 08:39:12 Rhäzüns Rennradfahren
Trackpunkte: 1451
Fertig: bikepacking_svg.html


In [ ]:
# ── SCHRITT 5 (optional): Etappendaten als JSON für die App exportieren ────────
# Falls Token-Upload nicht klappt: diese JSON-Datei in der App hochladen.
# App → Einstellungen → "Colab JSON importieren"
# ─────────────────────────────────────────────────────────────────────────────
import json

EXPORT_FILE = "bikepacking_colab_export.json"

stages_export = []
for s in stages:
    acts = s["acts"]
    avg_hr = [float(a["averageHR"]) for a in acts if a.get("averageHR")]
    max_hr = [float(a["maxHR"])     for a in acts if a.get("maxHR")]
    avg_pw = [float(a["averagePower"]) for a in acts if a.get("averagePower")]
    np_pw  = [float(a.get("normPower") or a.get("normalizedPower") or 0)
              for a in acts if a.get("normPower") or a.get("normalizedPower")]
    stages_export.append({
        "date":               s["day"],
        "title":              s["title"],
        "garmin_activity_id": ",".join(str(a["activityId"]) for a in acts if a.get("activityId")),
        "distance":           s["distance"],
        "elevation_gain":     s["elevation"],
        "moving_time":        int(sum(float(a.get("movingDuration")  or 0) for a in acts)),
        "elapsed_time":       int(sum(float(a.get("elapsedDuration") or 0) for a in acts)),
        "average_hr":         round(sum(avg_hr)/len(avg_hr), 1) if avg_hr else None,
        "max_hr":             max(max_hr) if max_hr else None,
        "average_power":      round(sum(avg_pw)/len(avg_pw), 1) if avg_pw else None,
        "normalized_power":   round(sum(np_pw)/len(np_pw),  1) if np_pw  else None,
        "load_score":         s["tss"],
        "track_points":       s["points"],
    })

payload = {
    "tour":   {"name": TOUR_NAME, "start_date": START_DATE, "description": "Garmin Export"},
    "stages": stages_export,
}
with open(EXPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f"✅ {len(stages_export)} Etappen exportiert")

from google.colab import files
files.download(EXPORT_FILE)
print("📥 bikepacking_colab_export.json heruntergeladen")